In [139]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [140]:
pd.set_option('display.max_columns',None)

In [141]:
url = 'https://raw.githubusercontent.com/ichiP245/my-next-soderling/refs/heads/main/Archivos/df_tenis_columns_OK.csv'

In [142]:
df = pd.read_csv(url)

In [143]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

In [144]:
df.columns

Index(['Unnamed: 0', 'ATP', 'Location', 'Tournament', 'Date', 'Series',
       'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank',
       'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4',
       'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'MaxW',
       'MaxL', 'AvgW', 'AvgL', 'Fecha', 'playerA', 'playerB', 'rankA', 'rankB',
       'PtsA', 'PtsB', 'B365A', 'B365B', 'MaxA', 'MaxB', 'AvgA', 'AvgB', 'A1',
       'B1', 'A2', 'B2', 'A3', 'B3', 'A4', 'B4', 'A5', 'B5', 'setsA', 'setsB',
       'target'],
      dtype='object')

In [145]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16061 entries, 0 to 16060
Data columns (total 61 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  16061 non-null  int64         
 1   ATP         16061 non-null  float64       
 2   Location    16061 non-null  object        
 3   Tournament  16061 non-null  object        
 4   Date        16061 non-null  object        
 5   Series      16061 non-null  object        
 6   Court       16061 non-null  object        
 7   Surface     16061 non-null  object        
 8   Round       16061 non-null  object        
 9   Best of     16061 non-null  float64       
 10  Winner      16061 non-null  object        
 11  Loser       16061 non-null  object        
 12  WRank       16061 non-null  float64       
 13  LRank       16061 non-null  float64       
 14  WPts        16061 non-null  float64       
 15  LPts        16061 non-null  float64       
 16  W1          16061 non-

## Creacion de Variables

Implied Probabilities de cada campo que tiene cuotas (odds)

In [146]:
def implied_prob_adjusted(odds_A, odds_B):
    """Pasamos odds decimales en probabilidades ajustadas eliminando el margen del bookmaker."""
    inv_A = 1 / odds_A
    inv_B = 1 / odds_B

    total = inv_A + inv_B

    prob_A = inv_A / total
    prob_B = inv_B / total

    return prob_A, prob_B

In [147]:
probA = 1 / df['B365A']
probB = 1 / df['B365B']
df['B365BookmakersMargin'] = (probA + probB - 1)

df['B365ProbA'], df['B365ProbB'] = implied_prob_adjusted(df['B365A'], df['B365B'])
df['ProbAvgA'], df['ProbAvgB'] = implied_prob_adjusted(df['AvgA'], df['AvgB'])
df['ProbMaxA'], df['ProbMaxB'] = implied_prob_adjusted(df['MaxA'], df['MaxB'])

df['market_uncertainty'] = np.abs(df['ProbAvgA'] - 0.5) # Mas bajo, mas parejo el partido

# Vemos la diferencia entre las probabilidades mas optimistas y las del consenso general del mercado
df['SpreadOddsA'] = df['ProbMaxA'] - df['ProbAvgA']
df['SpreadOddsB'] = df['ProbAvgB'] - df['ProbMaxB']

# Diferencia promedio de probabilidades
df['AvgOddsDiff'] = (df['ProbAvgA'] - df['ProbAvgB'])

# Probabilidades con un logaritmo aplicado, recomendado por ChatGPT
df['logit_oddsA'] = np.log(df['ProbAvgA'] / (1 - df['ProbAvgA']))
df['logit_oddsB'] = np.log(df['ProbAvgB'] / (1 - df['ProbAvgB']))

Ranking Difference, ATP Points Difference

In [148]:
df['rankDiff'] = df['rankA'] - df['rankB']
df['ptsDiff'] = df['PtsA'] - df['PtsB']

Porcentaje de partidos ganados en los últimos 5, 10 o 20 partidos

In [149]:
# @title
import pandas as pd
import numpy as np

def calcular_features_historial(df: pd.DataFrame, col_ganador: str = 'winner') -> pd.DataFrame:
    """
    Calcula features de historial por jugador para cada partido,
    usando SOLO información anterior al partido (sin data leakage).

    Parámetros:
    -----------
    df : DataFrame con columnas 'Fecha', 'playerA', 'playerB', y col_ganador
    col_ganador : nombre de la columna que indica quién ganó ('A' o 'B')

    Retorna:
    --------
    df original con nuevas columnas de features
    """

    df = df.copy()
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    df = df.sort_values('Fecha').reset_index(drop=True)

    # ------------------------------------------------------------------ #
    # PASO 1: Construir historial "largo" — una fila por jugador por partido
    # ------------------------------------------------------------------ #
    registros = []
    for _, row in df.iterrows():
        ganador = row['playerA'] if row['target'] == 1 else row['playerB']

        for rol, jugador, rival in [('A', row['playerA'], row['playerB']),
                                    ('B', row['playerB'], row['playerA'])]:
            sets_ganados  = row['setsA'] if rol == 'A' else row['setsB']
            sets_perdidos = row['setsB'] if rol == 'A' else row['setsA']

            registros.append({
                'fecha'        : row['Fecha'],
                'jugador'      : jugador,
                'gano'         : 1 if jugador == ganador else 0,
                'sets_jugados' : sets_ganados + sets_perdidos,
            })

    hist = pd.DataFrame(registros).sort_values('fecha').reset_index(drop=True)

    # ------------------------------------------------------------------ #
    # PASO 2: Calcular features por jugador con ventana histórica estricta
    #         (shift(1) garantiza que no usamos el partido actual)
    # ------------------------------------------------------------------ #
    def winrate_ultimos_n(group, n):
        """Winrate en los últimos N partidos anteriores al actual."""
        return (
            group['gano']
            .shift(1)                        # excluye el partido actual
            .rolling(window=n, min_periods=1)
            .mean()
        )

    def sets_ultimos_dias(group, dias):
        """Sets jugados en los últimos X días antes del partido actual."""
        resultados = []
        fechas     = group['fecha'].values
        sets       = group['sets_jugados'].values

        for i in range(len(group)):
            fecha_corte = fechas[i]
            limite      = fecha_corte - np.timedelta64(dias, 'D')
            # Suma sets de partidos ANTERIORES dentro de la ventana de días
            mask = (fechas < fecha_corte) & (fechas >= limite)
            resultados.append(sets[mask].sum())

        return resultados

    def partidos_ultimos_dias(group, dias):
        """Cantidad de partidos jugados en los últimos X días."""
        resultados = []
        fechas     = group['fecha'].values

        for i in range(len(group)):
            fecha_corte = fechas[i]
            limite      = fecha_corte - np.timedelta64(dias, 'D')
            mask        = (fechas < fecha_corte) & (fechas >= limite)
            resultados.append(mask.sum())

        return resultados

    def dias_desde_ultimo(group):
        """Días transcurridos desde el partido anterior."""
        return group['fecha'].diff().dt.days  # NaN para el primer partido

    # Aplicar por jugador
    features_list = []

    for jugador, group in hist.groupby('jugador'):
        group = group.copy().reset_index(drop=True)

        group['winrate_5']         = winrate_ultimos_n(group, 5)
        group['winrate_10']        = winrate_ultimos_n(group, 10)
        group['winrate_20']        = winrate_ultimos_n(group, 20)
        group['sets_5d']           = sets_ultimos_dias(group, 5)
        group['sets_30d']          = sets_ultimos_dias(group, 30)
        group['partidos_365d']     = partidos_ultimos_dias(group, 365)
        group['dias_ultimo_part']  = dias_desde_ultimo(group)

        features_list.append(group)

    hist_features = pd.concat(features_list).sort_values('fecha').reset_index(drop=True)

    # ------------------------------------------------------------------ #
    # PASO 3: Pegar las features de vuelta al dataframe original
    # ------------------------------------------------------------------ #
    FEATURE_COLS = [
        'winrate_5', 'winrate_10', 'winrate_20',
        'sets_5d', 'sets_30d', 'partidos_365d', 'dias_ultimo_part'
    ]

    def merge_features(df_orig, hist_feat, rol):
        """Mergea las features calculadas para playerA o playerB."""
        jugador_col = f'player{rol}'

        subset = hist_feat.rename(columns={
            'jugador': jugador_col,
            **{c: f'{c}_{rol}' for c in FEATURE_COLS}
        })[[jugador_col, 'fecha'] + [f'{c}_{rol}' for c in FEATURE_COLS]]

        return df_orig.merge(
            subset,
            left_on  = ['Fecha', jugador_col],
            right_on = ['fecha',  jugador_col],
            how      = 'left'
        ).drop(columns='fecha')

    df = merge_features(df, hist_features, 'A')
    df = merge_features(df, hist_features, 'B')

    return df


# ------------------------------------------------------------------ #
# USO
# ------------------------------------------------------------------ #
df_con_features = calcular_features_historial(df, col_ganador='winner')

In [150]:
jug = pd.concat([df['playerA'], df['playerB']], axis=0).value_counts()
print(f'Total: {jug.shape[0]} jugadores')
print(f'Con mas de 20 partidos: {jug[jug>20].shape[0]} jugadores')
print(f'Con mas de 10 partidos: {jug[jug>10].shape[0]} jugadores')
print(f'Con mas de 5 partidos: {jug[jug>5].shape[0]} jugadores')

Total: 613 jugadores
Con mas de 20 partidos: 264 jugadores
Con mas de 10 partidos: 325 jugadores
Con mas de 5 partidos: 385 jugadores


In [151]:
a = df[['Fecha','playerA','target', 'Unnamed: 0']].rename(columns={'playerA':'Player','target':'win'})
b = df[['Fecha','playerB','target', 'Unnamed: 0']].rename(columns={'playerB':'Player','target':'win'})
b['win'] = 1 - b['win']

In [152]:
matches_long = pd.concat([a[["Fecha", "Player", "win", 'Unnamed: 0']], b[["Fecha", "Player", "win", 'Unnamed: 0']]],
                         ignore_index=True).sort_values(["Fecha"]).reset_index(drop=True)

In [153]:
matches_long['winrate_5'] = (
    matches_long
    .groupby('Player')['win']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

matches_long['winrate_10'] = (
    matches_long
    .groupby('Player')['win']
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

In [154]:
matches_long["matches_played_before"] = matches_long.groupby("Player").cumcount()

matches_long["wins_before"] = (matches_long.groupby("Player")["win"].transform(lambda x: x.cumsum().shift(1)).fillna(0))

matches_long["win_pct_before"] = (matches_long["wins_before"] / matches_long["matches_played_before"].replace(0, pd.NA))

In [155]:
FEATURE_COLS = ['winrate_5', 'winrate_10', "wins_before", "win_pct_before"]

feats_A = (matches_long
    .merge(df[['Unnamed: 0', 'playerA']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerA'])
    [['Unnamed: 0'] + FEATURE_COLS]
    .rename(columns={c: f'{c}_A' for c in FEATURE_COLS})
)

feats_B = (matches_long
    .merge(df[['Unnamed: 0', 'playerB']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerB'])
    [['Unnamed: 0'] + FEATURE_COLS]
    .rename(columns={c: f'{c}_B' for c in FEATURE_COLS})
)

df = df.merge(feats_A, on='Unnamed: 0', how='left')
df = df.merge(feats_B, on='Unnamed: 0', how='left')

In [156]:
df.head(1)

,Unnamed: 0,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,Loser,WRank,LRank,WPts,LPts,W1,L1,W2,L2,W3,L3,W4,L4,W5,L5,Wsets,Lsets,Comment,B365W,B365L,MaxW,MaxL,AvgW,AvgL,Fecha,playerA,playerB,rankA,rankB,PtsA,PtsB,B365A,B365B,MaxA,MaxB,AvgA,AvgB,A1,B1,A2,B2,A3,B3,A4,B4,A5,B5,setsA,setsB,target,B365BookmakersMargin,B365ProbA,B365ProbB,ProbAvgA,ProbAvgB,ProbMaxA,ProbMaxB,market_uncertainty,SpreadOddsA,SpreadOddsB,AvgOddsDiff,logit_oddsA,logit_oddsB,rankDiff,ptsDiff,winrate_5_A,winrate_10_A,wins_before_A,win_pct_before_A,winrate_5_B,winrate_10_B,wins_before_B,win_pct_before_B
0,11805,3.0,Doha,Qatar Exxon Mobil Open,05/01/15,ATP250,Outdoor,Hard,1st Round,3.0,Gasquet R.,Andujar P.,26.0,41.0,1350.0,950.0,6.0,3.0,7.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,Completed,1.2,4.33,1.26,4.84,1.21,4.15,2015-01-05,Andujar P.,Gasquet R.,41.0,26.0,950.0,1350.0,4.33,1.2,4.84,1.26,4.15,1.21,3.0,6.0,5.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0,0.06428,0.216998,0.783002,0.225746,0.774254,0.206557,0.793443,0.274254,-0.019189,-0.019189,-0.548507,-1.232488,1.232488,15.0,-400.0,NaN,NaN,0.0,<NA>,NaN,NaN,0.0,<NA>


Sets jugados en ultimos 5 dias y sets jugados en ultimos 30 dias

In [157]:
df['setsPartido'] = df['setsA'] + df['setsB']

In [159]:
a = df[['Fecha','playerA','target', 'setsPartido', 'Unnamed: 0']].rename(columns={'playerA':'Player','target':'win'})
b = df[['Fecha','playerB','target', 'setsPartido', 'Unnamed: 0']].rename(columns={'playerB':'Player','target':'win'})
b['win'] = 1 - b['win']
matches_long = pd.concat([a, b], ignore_index=True).sort_values("Fecha").reset_index(drop=True)

In [161]:
def sets_ultimos_5_dias(group):
    resultados = []
    for i, (fecha, sets) in enumerate(zip(group['Fecha'], group['setsPartido'])):
        limite = fecha - pd.Timedelta(days=5)
        mask = (group['Fecha'] < fecha) & (group['Fecha'] >= limite)
        resultados.append(group.loc[mask, 'setsPartido'].sum())
    return pd.Series(resultados, index=group.index)

# Lo mismo que arriba pero cambiando la cantidad de dias
def sets_ultimos_30_dias(group):
    resultados = []
    for i, (fecha, sets) in enumerate(zip(group['Fecha'], group['setsPartido'])):
        limite = fecha - pd.Timedelta(days=30)
        mask = (group['Fecha'] < fecha) & (group['Fecha'] >= limite)
        resultados.append(group.loc[mask, 'setsPartido'].sum())
    return pd.Series(resultados, index=group.index)

matches_long['sets_5d'] = (
    matches_long.groupby('Player')
    .apply(sets_ultimos_5_dias)
    .reset_index(level=0, drop=True)
)
matches_long['sets_30d'] = (
    matches_long.groupby('Player')
    .apply(sets_ultimos_30_dias)
    .reset_index(level=0, drop=True)
)

/tmp/ipykernel_6347/3832201008.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(sets_ultimos_5_dias)
/tmp/ipykernel_6347/3832201008.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(sets_ultimos_30_dias)


In [163]:
FEATURE_COLS = ['sets_5d', 'sets_30d']

feats_A = (matches_long
    .merge(df[['Unnamed: 0', 'playerA']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerA'])
    [['Unnamed: 0'] + FEATURE_COLS]
    .rename(columns={c: f'{c}_A' for c in FEATURE_COLS})
)

feats_B = (matches_long
    .merge(df[['Unnamed: 0', 'playerB']], left_on=['Unnamed: 0', 'Player'], right_on=['Unnamed: 0', 'playerB'])
    [['Unnamed: 0'] + FEATURE_COLS]
    .rename(columns={c: f'{c}_B' for c in FEATURE_COLS})
)

df = df.merge(feats_A, on='Unnamed: 0', how='left')
df = df.merge(feats_B, on='Unnamed: 0', how='left')

Partidos en ultimos 365 dias

In [181]:
def partidos_ultimos_365_dias(group): # Renamed for clarity
  resultados = []
  for current_match_date in group['Fecha']:
    limite = current_match_date - pd.Timedelta(days=365)
    # The mask should look for matches strictly before the current_match_date
    mask = (group['Fecha'] < current_match_date) & (group['Fecha'] >= limite)
    resultados.append(group.loc[mask].shape[0]) # Count the number of rows (matches)
  return pd.Series(resultados, index=group.index)

In [182]:
matches_long['partidos_365d'] = matches_long.groupby('Player').apply(partidos_ultimos_365_dias, include_groups=False).reset_index(level=0, drop=True)

/tmp/ipykernel_6347/4277516507.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_long['partidos_365d'] = matches_long.groupby('Player').apply(partidos_ultimos_365_dias).reset_index(level=0, drop=True)


Dias desde el ultimo partido

In [183]:
def dias_desde_ultimo_partido(group):
    resultados = []
    for current_match_date in group['Fecha']:
        partidos_anteriores = group['Fecha'][group['Fecha'] < current_match_date]
        if partidos_anteriores.empty:
            resultados.append(pd.NA)  # Primer partido, no hay referencia
        else:
            resultados.append((current_match_date - partidos_anteriores.max()).days)
    return pd.Series(resultados, index=group.index)

In [184]:
matches_long['dias_ultimo_partido'] = matches_long.groupby('Player').apply(dias_desde_ultimo_partido, include_groups=False).reset_index(level=0, drop=True)

/tmp/ipykernel_6347/2411307702.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_long['dias_ultimo_partido']  = matches_long.groupby('Player').apply(dias_desde_ultimo_partido).reset_index(level=0, drop=True)


Diferencia de dias desde el ultimo partido (Jugador A - Jugador B)

Racha actual de victorias

In [185]:
def racha_victorias(group):
    resultados = []
    for i, current_match_date in enumerate(group['Fecha']):
        partidos_anteriores = group[group['Fecha'] < current_match_date].sort_values('Fecha', ascending=False)
        racha = 0
        for gano in partidos_anteriores['win']:
            if gano == 1:
                racha += 1
            else:
                break  # Se cortó la racha
        resultados.append(racha)
    return pd.Series(resultados, index=group.index)

In [186]:
matches_long['racha_victorias'] = matches_long.groupby('Player').apply(racha_victorias, include_groups=False).reset_index(level=0, drop=True)

/tmp/ipykernel_6347/2790891202.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  matches_long['racha_victorias']      = matches_long.groupby('Player').apply(racha_victorias).reset_index(level=0, drop=True)


Record en cada superficie, previo al partido

Cantidad de partidos previos entre ambos

In [ ]:
# Cantidad de partidos previos entre ambos jugadores (Head-to-Head)
# Create a unique identifier for each match pair, sorted alphabetically to handle both playerA vs playerB and playerB vs playerA
df['h2h_pair_id'] = df.apply(lambda row: tuple(sorted([row['playerA'], row['playerB']])), axis=1)

# Sort the DataFrame by this pair ID and then by date to ensure correct cumulative counting
df_sorted_h2h = df.sort_values(by=['h2h_pair_id', 'Fecha']).copy()

# Calculate the cumulative count of matches for each unique pair.
# cumcount() starts from 0, so it directly gives the number of previous matches for that pair.
df_sorted_h2h['h2h_matches_previous'] = df_sorted_h2h.groupby('h2h_pair_id').cumcount()

# Merge this new feature back into the original DataFrame using 'Unnamed: 0' (unique match ID)
df = df.merge(df_sorted_h2h[['Unnamed: 0', 'h2h_matches_previous']], on='Unnamed: 0', how='left')

# Drop the temporary column
df.drop(columns=['h2h_pair_id'], inplace=True)

## Encoding de categoricas

is_grand_slam
is_indoor
is_clay
is_best_of_5